# Fine-tune qilingan Whisper-small'ni qo'lda sinash (CPU yoki GPU, avtomatik)

Model Hugging Face'dan yuklanadi: `Toxirerkinovtohir/whisper-small-karakalpak` (birinchi marta ~970 MB). GPU bo'lsa GPU'da, bo'lmasa CPU'da ishlaydi.

1-katakni bir marta ishga tushiring, 2-katakni xohlagancha qayta bosing (har safar yangi klip). Bitta klip CPU'da ~5-20 soniya, GPU'da 1 soniyadan kam.

Dekodlash `transcribe.py` bilan bir xil: beam 5, `no_repeat` yo'q (07_eval'da validation bo'yicha tanlangan).

In [ ]:
# 1. Modelni yuklash (bir marta, ~10-20 soniya)
import time, torch, librosa, pandas as pd, jiwer
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration, logging
logging.set_verbosity_error()
torch.set_num_threads(max(1, torch.get_num_threads()))

MODEL = "Toxirerkinovtohir/whisper-small-karakalpak"   # fine-tune qilingan model (checkpoint-5000)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
processor = WhisperProcessor.from_pretrained(MODEL)
model = WhisperForConditionalGeneration.from_pretrained(MODEL).to(DEVICE).eval()   # float32

df = pd.read_csv("../04_normalize/manifest_tayyor.csv")
df["fayl"] = "../" + df["fayl"].str.replace("\\", "/", regex=False)   # yo'llar loyiha ildiziga nisbatan


def transkripsiya(fayl, beam=True):
    wav, _ = librosa.load(fayl, sr=16000)  # istalgan format / sample rate -> 16 kHz
    x = processor(wav, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
    kw = {"num_beams": 5} if beam else {}
    with torch.no_grad():
        out = model.generate(x, language="uz", task="transcribe", max_new_tokens=440, **kw)
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()


print("Model tayyor:", MODEL, "|", DEVICE)

## 2. Test to'plamidan tasodifiy klip

In [ ]:
# 2. Test to'plamidan tasodifiy klip: greedy va beam
r = df[df.split == "test"].sample(1).iloc[0]
display(Audio(r.fayl))
print("Fayl   :", r.fayl, f"({r.yangi_s:.1f} s)")
print("HAQIQIY:", r.matn_model)
for beam in [False, True]:
    t0 = time.time()
    t = transkripsiya(r.fayl, beam=beam)
    nom = "BEAM   " if beam else "GREEDY "
    print(f"{nom}: {t}")
    print(f"         WER {100 * jiwer.wer(r.matn_model, t):.1f}%   CER {100 * jiwer.cer(r.matn_model, t):.1f}%   ({time.time() - t0:.1f} s)")

## 3. O'z audiongiz

In [ ]:
# 3. O'z audiongiz: fayl yo'lini yozing (repo ildiziga nisbatan)
import soundfile as sf

FAYL = "../DATASET_trimmed/test/0.flac"
ovoz, sr = sf.read(FAYL)
if ovoz.ndim > 1:
    ovoz = ovoz.T
display(Audio(ovoz, rate=sr))
print(transkripsiya(FAYL))